# Data Cleanup for D8 Startup Dataset

This notebook creates a **compact research subset** from `data/Startup_Data.csv`. The target is to keep **at most 20 columns total**, not 90+. The retained set is designed to cover the project's main hypotheses and a small number of controls for funding, founder quality, team structure, business model, and market traction.


## Retention logic

The final subset keeps 20 columns in total:

- `Dependent-Company Status` as the outcome.
- Main hypothesis variables: `Number of Co-founders`, `B2C or B2B venture?`, `Time to 1st investment (in months)`.
- Team and traction controls: advisor count, senior leadership size, employee growth, internet activity.
- Funding controls: last funding round size, investor participation, repeat investors, top-fund presence.
- Founder quality controls: founder experience, prior startup success, university tier.
- Business model controls: linear vs non-linear and capital intensity.
- Market and cohort controls: founding year and industry investment trend.
- One technology signal: `Disruptiveness of technology`.

Everything else is removed because it is either raw text, too redundant, too noisy, too sparse, too high-cardinality for this sample size, or outside the core econometric questions.


In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("data/Startup_Data.csv")
CLEAN_PATH = Path("data/Startup_Data_research_subset.csv")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

df = pd.read_csv(RAW_PATH, encoding="latin1")
print(f"Raw shape: {df.shape}")
df.head(3)


Raw shape: (472, 116)


,Company_Name,Dependent-Company Status,year of founding,Age of company in years,Internet Activity Score,Short Description of company profile,Industry of company,Focus functions of company,Investors,Employee Count,Employees count MoM change,Has the team size grown,Est. Founding Date,Last Funding Date,Last Funding Amount,Country of company,Continent of company,Number of Investors in Seed,Number of Investors in Angel and or VC,Number of Co-founders,Number of of advisors,Team size Senior leadership,Team size all employees,Presence of a top angel or venture fund in previous round of investment,Number of of repeat investors,Number of Sales Support material,Worked in top companies,Average size of companies worked for in the past,Have been part of startups in the past?,Have been part of successful startups in the past?,Was he or she partner in Big 5 consulting?,Consulting experience?,Product or service company?,Catering to product/service across verticals,Focus on private or public data?,Focus on consumer data?,Focus on structured or unstructured data,Subscription based business,Cloud or platform based serive/product?,Local or global player,Linear or Non-linear business model,"Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive",Number of of Partners of company,Crowdsourcing based business,Crowdfunding based business,Machine Learning based business,Predictive Analytics business,Speech analytics business,Prescriptive analytics business,Big Data Business,Cross-Channel Analytics/ marketing channels,Owns data or not? (monetization of data) e.g. Factual,Is the company an aggregator/market place? e.g. Bluekai,Online or offline venture - physical location based business or online venture?,B2C or B2B venture?,Top forums like 'Tech crunch' or 'Venture beat' talking about the company/model - How much is it being talked about?,Average Years of experience for founder and co founder,Exposure across the globe,Breadth of experience across verticals,Highest education,Years of education,Specialization of highest education,Relevance of education to venture,Relevance of experience to venture,Degree from a Tier 1 or Tier 2 university?,Renowned in professional circle,Experience in selling and building products,Experience in Fortune 100 organizations,Experience in Fortune 500 organizations,Experience in Fortune 1000 organizations,Top management similarity,Number of Recognitions for Founders and Co-founders,Number of of Research publications,Skills score,Team Composition score,Dificulty of Obtaining Work force,Pricing Strategy,Hyper localisation,Time to market service or product,Employee benefits and salary structures,Long term relationship with other founders,Proprietary or patent position (competitive position),Barriers of entry for the competitors,Company awards,Controversial history of founder or co founder,Legal risk and intellectual property,Client Reputation,google page rank of company website,Technical proficiencies to analyse and interpret unstructured data,Solutions offered,Invested through global incubation competitions?,Industry trend in investing,Disruptiveness of technology,Number of Direct competitors,Employees per year of company existence,Last round of funding received (in milionUSD),"Survival through recession, based on existence of the company through recession times",Time to 1st investment (in months),"Avg time to investment - average across all rounds, measured from previous investment",Gartner hype cycle stage,Time to maturity of technology (in years),Percent_skill_Entrepreneurship,Percent_skill_Operations,Percent_skill_Engineering,Percent_skill_Marketing,Percent_skill_Leadership,Percent_skill_Data Science,Percent_skill_Business Strategy,Percent_skill_Product Management,Percent_skill_Sales,Percent_skill_Domain,Percent_skill_Law,Percent_skill_Consulting,Percent_skill_Finance,Percent_skill_Investment,Renown score
0,Company1,Success,No Info,No Info,-1.0,Video di

## Standardize obvious categorical inconsistencies

The raw file mixes labels such as `YES`, `yes`, `Yes`, lowercase intensity levels, blank strings, and `No Info`. This step cleans those inconsistencies first.


In [2]:
def normalize_string_column(series: pd.Series) -> pd.Series:
    if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
        return series

    series = series.astype("string").str.strip()
    replacements = {
        "YES": "Yes",
        "yes": "Yes",
        "NO": "No",
        "high": "High",
        "medium": "Medium",
        "low": "Low",
        "No info": "No Info",
    }
    return series.replace(replacements)


df = df.apply(normalize_string_column)

numeric_like_columns = []
for col in df.columns:
    non_missing = df[col].dropna()
    if non_missing.empty:
        continue

    converted = pd.to_numeric(
        non_missing.astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    if converted.notna().all():
        numeric_like_columns.append(col)

for col in numeric_like_columns:
    df[col] = pd.to_numeric(
        df[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

print(f"Numeric-like columns converted: {len(numeric_like_columns)}")


Numeric-like columns converted: 8


## Keep only the research subset

These columns are retained on purpose. This is the compact dataset that should be used for the next stage of modeling.


In [3]:
keep_groups = {
    "Outcome": [
        "Dependent-Company Status",
    ],
    "Main hypotheses": [
        "Number of Co-founders",
        "B2C or B2B venture?",
        "Time to 1st investment (in months)",
    ],
    "Team and traction controls": [
        "Number of of advisors",
        "Team size Senior leadership",
        "Employees count MoM change",
        "Internet Activity Score",
    ],
    "Funding controls": [
        "Last round of funding received (in milionUSD)",
        "Number of Investors in Angel and or VC",
        "Number of of repeat investors",
        "Presence of a top angel or venture fund in previous round of investment",
    ],
    "Founder quality controls": [
        "Average Years of experience for founder and co founder",
        "Have been part of successful startups in the past?",
        "Degree from a Tier 1 or Tier 2 university?",
    ],
    "Business model controls": [
        "Linear or Non-linear business model",
        "Capital intensive business e.g. e-commerce, Engineering products and operations can also cause a business to be capital intensive",
    ],
    "Market and technology controls": [
        "year of founding",
        "Industry trend in investing",
        "Disruptiveness of technology",
    ],
}

keep_columns = [col for cols in keep_groups.values() for col in cols]
missing_from_source = [col for col in keep_columns if col not in df.columns]
assert not missing_from_source, f"Columns not found: {missing_from_source}"
assert len(keep_columns) <= 20, f"Too many retained columns: {len(keep_columns)}"

retained = pd.Series(
    {col: group for group, cols in keep_groups.items() for col in cols},
    name="group",
).rename_axis("column").reset_index()

print(f"Retained columns: {len(keep_columns)}")
retained


Retained columns: 20


,column,group
0,Dependent-Company Status,Outcome
1,Number of Co-founders,Main hypotheses
2,B2C or B2B venture?,Main hypotheses
3,Time to 1st investment (in months),Main hypotheses
4,Number of of advisors,Team and traction controls
5,Team size Senior leadership,Team and traction controls
6,Employees count MoM change,Team and traction controls
7,Internet Activity Score,Team and traction controls
8,Last round of funding received (in milionUSD),Funding controls
9,Number of Investors in Angel and or VC,Funding controls


In [4]:
df_clean = df[keep_columns].copy()

print(f"Raw shape: {df.shape}")
print(f"Research subset shape: {df_clean.shape}")
print(f"Columns removed: {df.shape[1] - df_clean.shape[1]}")
print(f"Columns retained: {df_clean.shape[1]}")


Raw shape: (472, 116)
Research subset shape: (472, 20)
Columns removed: 96
Columns retained: 20


## Quick audit of the research subset

This check shows missingness and cardinality in the final retained variables so the next notebook can decide how to encode and impute them.


In [5]:
audit = pd.DataFrame(
    {
        "dtype": df_clean.dtypes.astype(str),
        "missing_pct": (df_clean.isna().mean() * 100).round(1),
        "nunique": df_clean.nunique(dropna=True),
    }
).sort_values(["missing_pct", "nunique"], ascending=[False, False])

audit


,dtype,missing_pct,nunique
Employees count MoM change,Float64,43.4,41
Degree from a Tier 1 or Tier 2 university?,string,30.5,4
Industry trend in investing,Float64,17.4,6
Internet Activity Score,Float64,13.8,260
Last round of funding received (in milionUSD),string,0.0,142
Time to 1st investment (in months),string,0.0,52
year of founding,string,0.0,16
Team size Senior leadership,Int64,0.0,14
Number of of advisors,Int64,0.0,13
Number of Investors in Angel and or VC,string,0.0,11


In [6]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved research subset to: {CLEAN_PATH}")


Saved research subset to: data/Startup_Data_research_subset.csv


## Next step

The next notebook should handle modeling preprocessing only: encode binary and ordinal variables, decide how to treat `No Info` and missing values, and then estimate the compact logit specification.
